In [3]:
def clean_data(df):

    df = df.copy()

    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["Churn"] = df["Churn"].map({"Yes":1, "No":0})

    df = df.drop("customerID", axis=1)

    df = df.dropna()

    return df

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", None)

df = pd.read_csv('../../data/raw/telco_customer_churn.csv')

df = clean_data(df)

In [5]:
print(df.head())

   gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  Female              0     Yes         No       1           No   
1    Male              0      No         No      34          Yes   
2    Male              0      No         No       2          Yes   
3    Male              0      No         No      45           No   
4  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity OnlineBackup  \
0  No phone service             DSL             No          Yes   
1                No             DSL            Yes           No   
2                No             DSL            Yes          Yes   
3  No phone service             DSL            Yes           No   
4                No     Fiber optic             No           No   

  DeviceProtection TechSupport StreamingTV StreamingMovies        Contract  \
0               No          No          No              No  Month-to-month   
1              Yes          No  

In [6]:
print(df.info())

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   str    
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   str    
 3   Dependents        7032 non-null   str    
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   str    
 6   MultipleLines     7032 non-null   str    
 7   InternetService   7032 non-null   str    
 8   OnlineSecurity    7032 non-null   str    
 9   OnlineBackup      7032 non-null   str    
 10  DeviceProtection  7032 non-null   str    
 11  TechSupport       7032 non-null   str    
 12  StreamingTV       7032 non-null   str    
 13  StreamingMovies   7032 non-null   str    
 14  Contract          7032 non-null   str    
 15  PaperlessBilling  7032 non-null   str    
 16  PaymentMethod     7032 non-null   str    
 17  MonthlyChar

In [7]:
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [8]:
print("Data duplicate:", df.duplicated().sum())

Data duplicate: 22


In [ ]:
# Categorical data quality overview + rule-based validation
categorical_cols = df.select_dtypes(include=["string", "object", "category"]).columns

summary_rows = []
for col in categorical_cols:
    s = df[col].astype("string").str.strip()
    summary_rows.append({
        "column": col,
        "rows": len(s),
        "missing": int(s.isna().sum()),
        "empty_string": int((s == "").sum()),
        "unique_values": int(s.nunique(dropna=True)),
        "top_5_values": s.value_counts(dropna=False).head(5).to_dict()
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["missing", "empty_string", "unique_values"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("Categorical Quality Summary")
display(summary_df)

# Allowed values for business-rule checks (add more columns as needed)
allowed_values = {
    "gender": {"Male", "Female"},
    "Partner": {"Yes", "No"},
    "Dependents": {"Yes", "No"},
    "PhoneService": {"Yes", "No"},
    "MultipleLines": {"Yes", "No", "No phone service"},
    "InternetService": {"DSL", "Fiber optic", "No"},
    "Contract": {"Month-to-month", "One year", "Two year"},
    "PaperlessBilling": {"Yes", "No"},
}

validation_rows = []
for col, allowed in allowed_values.items():
    if col in df.columns:
        s = df[col].astype("string").str.strip()
        actual = set(s.dropna().unique())
        unexpected = sorted(actual - allowed)
        validation_rows.append({
            "column": col,
            "is_valid": len(unexpected) == 0,
            "unexpected_values": unexpected
        })

validation_df = pd.DataFrame(validation_rows)
print("\nBusiness Rule Validation")
display(validation_df)

Categorical Quality Summary


,column,rows,missing,empty_string,unique_values,top_5_values
0,PaymentMethod,7032,0,0,4,"{'Electronic check': 2365, 'Mailed check': 160..."
1,MultipleLines,7032,0,0,3,"{'No': 3385, 'Yes': 2967, 'No phone service': ..."
2,InternetService,7032,0,0,3,"{'Fiber optic': 3096, 'DSL': 2416, 'No': 1520}"
3,OnlineSecurity,7032,0,0,3,"{'No': 3497, 'Yes': 2015, 'No internet service..."
4,OnlineBackup,7032,0,0,3,"{'No': 3087, 'Yes': 2425, 'No internet service..."
5,DeviceProtection,7032,0,0,3,"{'No': 3094, 'Yes': 2418, 'No internet service..."
6,TechSupport,7032,0,0,3,"{'No': 3472, 'Yes': 2040, 'No internet service..."
7,StreamingTV,7032,0,0,3,"{'No': 2809, 'Yes': 2703, 'No internet service..."
8,StreamingMovies,7032,0,0,3,"{'No': 2781, 'Yes': 2731, 'No internet service..."
9,Contract,7032,0,0,3,"{'Month-to-month': 3875, 'Two year': 1685, 'On..."



Business Rule Validation


,column,is_valid,unexpected_values
0,gender,True,[]
1,Partner,True,[]
2,Dependents,True,[]
3,PhoneService,True,[]
4,MultipleLines,True,[]
5,InternetService,True,[]
6,Contract,True,[]
7,PaperlessBilling,True,[]


## Data Dictionary (Business-Friendly)

- `customerID`: รหัสลูกค้า (unique identifier)
- `gender`: เพศของลูกค้า (Male/Female)
- `SeniorCitizen`: สถานะผู้สูงอายุ (1 = Yes, 0 = No)
- `Partner`: มีคู่สมรส/คู่ชีวิตหรือไม่ (Yes/No)
- `Dependents`: มีผู้ที่ต้องดูแลหรือไม่ (Yes/No)
- `tenure`: จำนวนเดือนที่เป็นลูกค้า
- `PhoneService`: มีบริการโทรศัพท์หรือไม่ (Yes/No)
- `MultipleLines`: มีหลายเบอร์หรือไม่ (Yes/No/No phone service)
- `InternetService`: ประเภทอินเทอร์เน็ต (DSL/Fiber optic/No)
- `OnlineSecurity`: มีบริการความปลอดภัยออนไลน์หรือไม่
- `OnlineBackup`: มีบริการสำรองข้อมูลออนไลน์หรือไม่
- `DeviceProtection`: มีบริการคุ้มครองอุปกรณ์หรือไม่
- `TechSupport`: มีบริการช่วยเหลือทางเทคนิคหรือไม่
- `StreamingTV`: มีบริการดูทีวีสตรีมมิงหรือไม่
- `StreamingMovies`: มีบริการดูภาพยนตร์สตรีมมิงหรือไม่
- `Contract`: ประเภทสัญญา (Month-to-month/One year/Two year)
- `PaperlessBilling`: ใช้ใบแจ้งหนี้อิเล็กทรอนิกส์หรือไม่
- `PaymentMethod`: วิธีชำระเงินของลูกค้า
- `MonthlyCharges`: ค่าใช้บริการรายเดือน
- `TotalCharges`: ค่าใช้บริการสะสม
- `Churn`: สถานะการยกเลิกบริการ (Yes/No)

หมายเหตุ: นิยามตัวแปรนี้ใช้เพื่อ align ความเข้าใจระหว่างทีมธุรกิจและทีมวิเคราะห์.

In [10]:
#Count churn amount
churn_counts = df['Churn'].value_counts()
print("\nChurn Value Counts:")
print(churn_counts)


Churn Value Counts:
Churn
0    5163
1    1869
Name: count, dtype: int64


In [15]:
# Check churn distribution
no_count = churn_counts.get(0, churn_counts.get('No', 0))
yes_count = churn_counts.get(1, churn_counts.get('Yes', 0))

if yes_count == 0:
    print("\nChurn Imbalance Ratio : N/A (no churned customers)")
else:
    churn_imbalance_ratio = no_count / yes_count
    print(f"\nChurn Imbalance Ratio : {churn_imbalance_ratio:.2f}")

# Check churn rate (%)
churn_rate = (yes_count / churn_counts.sum()) * 100
print(f"Churn Rate : {churn_rate:.2f}%")


Churn Imbalance Ratio : 2.76
Churn Rate : 26.58%


### Overview Conclusion

- Dataset มีลูกค้า **7,043 ราย** และ churn rate ประมาณ **26.5%**
- ไม่พบ duplicate row ในการตรวจเบื้องต้น
- ข้อมูลพร้อมสำหรับขั้นตอน **Data Quality** และ **Data Preparation**

**Next step:** เปิด [02_data_quality.ipynb](02_data_quality.ipynb) เพื่อทำ quality validation อย่างเป็นระบบ.